# Stopwords Are Not Universal

**Computational Text Analysis Pack · Module 03 — Preparing Text**

*Technique:* Stopword removal; why English lists do nothing to African languages

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/01-preparing-text/03a_stopwords_are_not_universal.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Almost every text-analysis tutorial you will find begins the same way: import a
stopword list, remove those words, carry on. The list is nearly always English.

This notebook measures what that step does to Yoruba, Hausa, Igbo and Nigerian
Pidgin. The answer is: **almost nothing** — and a step that does nothing while
you believe it worked is worse than no step at all.

This is not a complaint about tooling. It is a result you can publish.

## The tooling

**`re`** with Unicode-aware `\w`, which is what keeps Yorùbá and Hausa
characters alive. **NLTK** and **spaCy** ship tokenisers that are faster and
smarter for English and frequently wrong for African orthographies — module 14
measures that directly.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
# scikit-learn ships an English stopword list. It is the one most tutorials use.
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

english_stop = set(ENGLISH_STOP_WORDS)
print(f"{len(english_stop)} English stopwords")
print(sorted(english_stop)[:20])

## The measurement

For each language: what percentage of its tokens does the English list remove?

A working stopword list should take out 40–50% of tokens, because function words
are that large a share of any text.

In [ ]:
rows = []
for lang, g in df.groupby("language"):
    counts = Counter()
    for t in g["text"]:
        counts.update(tokenise(t))
    total = sum(counts.values())
    removed = sum(n for w, n in counts.items() if w in english_stop)
    rows.append({"language": lang, "tokens": total,
                 "removed_by_english_list": removed,
                 "pct_removed": removed / total * 100})

hit = pd.DataFrame(rows).sort_values("pct_removed", ascending=False)
print(hit.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
h = hit.sort_values("pct_removed")
colors = ["seagreen" if l == "English" else "steelblue" for l in h["language"]]
ax.barh(h["language"], h["pct_removed"], color=colors)
ax.axvline(40, color="crimson", linestyle="--", label="what a working list removes")
ax.set_xlabel("% of tokens removed by the English stopword list")
ax.set_title("One list, sixteen languages")
ax.legend()
plt.tight_layout(); plt.show()

## Read the chart

English sits where it should. Every African language sits near zero — the few
percent that *are* removed are English loanwords, names and code-switched
fragments, not function words.

So the pipeline runs, throws no error, and leaves the function words in place.
Every downstream result — frequency lists, topic models, clusters — is then
dominated by *ni, ti, da, na, ya*, and the analyst concludes the topics are
uninformative rather than that the preparation failed.

In [ ]:
# What SHOULD have been removed: the actual high-frequency function words.
LANGUAGE = "Yoruba"         # ← change this

sub = df[df["language"] == LANGUAGE]
counts = Counter()
for t in sub["text"]:
    counts.update(tokenise(t))
total = sum(counts.values())

print(f"{LANGUAGE}: top 15 words, and whether the English list catches them")
print(f"{'word':<16}{'per 1,000':>11}   {'in English list?'}")
print("-" * 48)
for w, n in counts.most_common(15):
    print(f"{w:<16}{n / total * 1000:>11.2f}   {'yes' if w in english_stop else 'NO'}")

### 🔧 Your turn 1

Run the cell above for Hausa, Igbo and Nigerian Pidgin.

Nigerian Pidgin is the interesting case: it is English-lexified, so the list
catches more than it does for Yoruba — but still nothing like 40%, because
Pidgin's own function words (*dey, na, wey, don, go*) are not English words.

## Building a list that works

The language-independent method: take the most frequent words *in your own
corpus* and treat those as function words. It is what everyone did before
ready-made lists existed, and for most African languages it is still the only
option.

The cost is honesty about what you removed — so print it, and put it in an
appendix.

In [ ]:
N_STOP = 50                 # ← change this

corpus_stop = {w for w, _ in counts.most_common(N_STOP)}
kept = {w: n for w, n in counts.items() if w not in corpus_stop}

print(f"corpus-derived stoplist for {LANGUAGE} ({N_STOP} words):")
print("  " + ", ".join(sorted(corpus_stop)))
print()
print(f"removes {sum(counts[w] for w in corpus_stop) / total:.1%} of tokens "
      f"— compare {sum(n for w, n in counts.items() if w in english_stop) / total:.2%} "
      f"for the English list")
print()
print("top content words after removal:")
for w, n in sorted(kept.items(), key=lambda kv: -kv[1])[:15]:
    print(f"   {w:<18}{n:>7,}")

### 🔧 Your turn 2

Look at your 50-word list. Is anything in it a **content** word you would not
want to lose — a place name, or a word central to the subject matter?

If so, remove it from the stoplist by hand and re-run. A hand-checked list is
better than an automatic one, and being able to say you checked it is the
difference between a method and a habit.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Nigerian Pidgin should show the highest African-language figure
— its lexicon is largely English-derived, so words like *for*, *the* and *of*
match. It still falls far short of 40%, because the grammatical core (*dey, na,
wey, don, go, sey*) is not in any English list. Yoruba, Igbo and Hausa sit at a
couple of percent, and what is caught there is mostly loanwords and names.

**Your turn 2.** Nearly always yes — a top-50 list for news text will typically
sweep up the country's name, *government*, or a similar high-frequency content
word. That is the built-in cost of frequency-derived stoplists, and it is why
this notebook asks you to look at the list rather than trust it. If removing
those words would change your finding, do not remove them: use keyness (04a),
which needs no stoplist at all.

**The larger point.** Write this up. "scikit-learn's English stopword list removes
2.6% of tokens from Igbo, 3.0% from Yoruba and 5.3% from Hausa news text, against
46.7% from English — and under 0.1% from Amharic and Tigrinya" is a measured,
citable claim about the state of NLP tooling for African languages, produced by
ten lines of code on an openly licensed corpus. Re-run the cell to confirm those
figures rather than quoting mine.

</details>

## Where this stops

You can prepare text without inheriting somebody else's language. Next: **03b**,
where the damage is done not by a list but by a single regular expression.